# CriterionLens validation evaluation and benchmarks

This job evaluates only the fixed validation split and benchmarks the merged models. It must not be used for final test evaluation.

In [ ]:
import os
from pathlib import Path
import shutil
import subprocess
import sys
import torch

devices = [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]
print({'devices': devices, 'torch': torch.__version__, 'architectures': torch.cuda.get_arch_list()})
if len(devices) < 2 or any('T4' not in name for name in devices):
    raise RuntimeError(f'This evaluation job requires two Tesla T4 GPUs; found {devices}')

candidates = sorted({path.parent for path in Path('/kaggle/input').rglob('requirements.txt') if (path.parent / 'pyproject.toml').is_file()})
if len(candidates) != 1:
    raise RuntimeError(f'Expected exactly one code dataset with requirements.txt and pyproject.toml; found {candidates}')
DATASET_DIR = candidates[0]
print(f'Found code dataset at: {DATASET_DIR}')
CODE_DIR = Path('/kaggle/working/code')
if CODE_DIR.exists():
    shutil.rmtree(CODE_DIR)
shutil.copytree(DATASET_DIR, CODE_DIR, ignore=shutil.ignore_patterns('__pycache__', '*.pyc'))
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', str(CODE_DIR / 'requirements.txt')], check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', '.', '--no-deps'], cwd=CODE_DIR, check=True)

from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version
try:
    torchao_version = version('torchao')
except PackageNotFoundError:
    torchao_version = None
if torchao_version and Version(torchao_version) < Version('0.16.1'):
    subprocess.run([sys.executable, '-m', 'pip', 'uninstall', '-y', 'torchao'], check=True)

OUT_DIR = Path('/kaggle/working/out')
OUT_DIR.mkdir(parents=True, exist_ok=True)
os_env = os.environ.copy()
os_env['PYTHONPATH'] = str(CODE_DIR / 'src') + os.pathsep + os_env.get('PYTHONPATH', '')
print(f'Code directory: {CODE_DIR}; output directory: {OUT_DIR}')

In [ ]:
import concurrent.futures

evaluation_runs = [
    ('base_1p5b', 'Qwen/Qwen2.5-1.5B-Instruct', None, '0'),
    ('base_0p5b', 'Qwen/Qwen2.5-0.5B-Instruct', None, '1'),
    ('ft_1p5b', 'Qwen/Qwen2.5-1.5B-Instruct', 'Ashher123/criterion-lens-qwen2.5-1.5b-lora', '0'),
    ('ft_0p5b', 'Qwen/Qwen2.5-0.5B-Instruct', 'Ashher123/criterion-lens-qwen2.5-0.5b-lora', '1'),
]

def run_evaluation(item):
    name, model_id, adapter_id, gpu = item
    command = [sys.executable, 'scripts/evaluate.py', '--model', model_id, '--split', 'val', '--decoding', 'no_cache', '--out', str(CODE_DIR / 'results' / 'eval' / f'val_{name}')]
    if adapter_id:
        command.extend(['--adapter', adapter_id])
    environment = os_env.copy()
    environment['CUDA_VISIBLE_DEVICES'] = gpu
    result = subprocess.run(command, cwd=CODE_DIR, env=environment, check=False)
    if result.returncode:
        raise subprocess.CalledProcessError(result.returncode, command)
    return name

for pair in (evaluation_runs[:2], evaluation_runs[2:]):
    with concurrent.futures.ThreadPoolExecutor(max_workers=2) as executor:
        for name in executor.map(run_evaluation, pair):
            print(f'Completed validation evaluation: {name}')

In [ ]:
benchmark_commands = [
    [sys.executable, 'scripts/bench_kv.py', '--config', 'configs/bench.yaml', '--model', model, '--output-dir', str(CODE_DIR / 'results' / 'bench')]
    for model in ('1.5B', '0.5B', 'tiny')
]
benchmark_commands.extend([
    [sys.executable, 'scripts/bench_attention.py', '--config', 'configs/attention_bench.yaml', '--output-dir', str(CODE_DIR / 'results' / 'bench')],
    [sys.executable, 'scripts/bench_speculative.py', '--config', 'configs/speculative_bench.yaml', '--prompt-source', 'val', '--draft', 'base_0.5B', '--output-dir', str(CODE_DIR / 'results' / 'bench')],
    [sys.executable, 'scripts/bench_speculative.py', '--config', 'configs/speculative_bench.yaml', '--prompt-source', 'val', '--draft', 'finetuned_0.5B', '--output-dir', str(CODE_DIR / 'results' / 'bench')],
])
for command in benchmark_commands:
    subprocess.run(command, cwd=CODE_DIR, env=os_env, check=True)
    print(f'Completed benchmark: {command[1:3]}')

import zipfile
archive_path = OUT_DIR / 'val_bench_results.zip'
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for relative_root in (Path('results/eval'), Path('results/bench')):
        root = CODE_DIR / relative_root
        if root.exists():
            for path in root.rglob('*'):
                if path.is_file():
                    archive.write(path, path.relative_to(CODE_DIR))
print(f'Packaged validation and benchmark results at {archive_path}')